# Programación - FakeFinder: Mejorando la Expedición con Aprendizaje por Transferencia

¡Bienvenido de nuevo, Explorador de IA! En tu primera expedición, construiste con éxito un **FakeFinder** desde cero, navegando por el complejo paisaje de los hiperparámetros con Optuna para diseñar una arquitectura CNN personalizada. Aprendiste los principios fundamentales de la construcción y optimización de modelos, una habilidad esencial para cualquier profesional del aprendizaje profundo.

Ahora, es el momento de actualizar tu kit de herramientas para el siguiente tramo del viaje. En el mundo real, los exploradores no siempre construyen sus vehículos desde cero; a menudo modernizan un motor potente y preexistente para lograr un rendimiento superior con mayor eficiencia. Esa es tu misión en esta tarea: aprovechar el poder del **aprendizaje por transferencia (transfer learning)**.

Utilizarás el conocimiento de un modelo veterano, **MobileNetV3-Large**, que ya ha sido entrenado en el vasto conjunto de datos ImageNet. Al adaptar este modelo preentrenado a nuestro conjunto de datos específico `AIvsReal`, verás una mejora dramática en el rendimiento y la velocidad.

A lo largo de esta expedición mejorada:

* Prepararás tu conjunto de datos de imágenes familiar con procesos de transformación optimizados para modelos preentrenados.
* Comprenderás los principios básicos del aprendizaje por transferencia y su papel fundamental en la visión artificial moderna.
* Cargarás un modelo potente preentrenado y lo adaptarás para una nueva tarea utilizando la **extracción de características**.
* Congelarás la base convolucional y reemplazarás la capa clasificadora final para especializar el modelo en la detección de imágenes falsas.
* Entrenarás tu modelo adaptado y compararás sus notables resultados con el modelo personalizado de tu primera tarea.



Prepárate para ver cómo aprovechar la sabiduría preentrenada puede acelerar tu camino hacia la construcción de modelos altamente efectivos y de vanguardia.

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents
- [Imports](#0)
- [1 - Preparing for the Next Expedition: A Quick Reconnaissance](#1)
    - [1.1 - Scouting the Archives: Loading Data with ImageFolder](#1-1)
        - **[Exercise 1 - create_dataset_splits](#ex-1)**    
    - [1.2 - Specialized Augmentations: Protocols for Training and Validation](#1-2)
        - **[Exercise 2 - define_transformations](#ex-2)**
    - [1.3 - Preparing the Data Launchpad: Applying Transformations and Batching Data](#1-3)
        - **[Exercise 3 - create_data_loaders](#ex-3)**    
- [2 - Advanced Expedition: Leveraging Pre-trained Model Wisdom](#2)
    - [2.1 - Selecting Your Champion: Introducing MobileNetV3-Large](#2-1)
        - **[Exercise 4.1 - load_mobilenetv3_model](#ex-41)**
    - [2.2 - Tailoring Your Champion: Adapting MobileNetV3 for Fake Detection](#2-2)
        - **[Exercise 4.2 - update_model_last_layer](#ex-42)**
- [3 - Final Expedition: Training and Evaluating Your Adapted Champion](#3)
    - [3.1 - Setting the Launchpad: Preparing DataLoaders and the Adapted Model](#3-1)
    - [3.2 - Training the Adapted Model](#3-2)
    - [3.3 - Performance Review: Visualizing Predictions](#3-3)

<a name='0'></a>
## Imports

In [ ]:
import numpy
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as tv_models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

In [ ]:
import helper_utils
import unittests

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {DEVICE}")

<a name='1'></a>
## 1 - Preparación para la próxima expedición: Un reconocimiento rápido

Esta tarea continúa la anterior, utilizando la misma colección curada de imágenes del conjunto de datos [AI-Generated Images vs Real Images](https://www.kaggle.com/datasets/tristanzhang32/ai-generated-images-vs-real-images). 

Como recordatorio, este conjunto de datos es una muestra de una colección más grande de 60,000 imágenes. La mitad son generadas por modelos de IA avanzados (Stable Diffusion, MidJourney y DALL·E), y la otra mitad son imágenes auténticas de fuentes como Pexels y Unsplash. Trabajarás con el mismo subconjunto cuidadosamente seleccionado de **5,000 imágenes para entrenamiento** y **1,000 para prueba/validación**.

La organización de los datos es la misma que antes. Las imágenes están clasificadas en carpetas `train` (entrenamiento) y `test` (prueba), cada una con subdirectorios `real` y `fake`. Es fundamental comprender esta estructura lógica, ya que influye directamente en cómo cargarás los datos para las próximas tareas.

Aquí tienes un recordatorio de la disposición de los datos:

```
./AIvsReal_sampled/
└───train/
|   ├───fake/
|   │   ├───0046.jpg
|   │   └───...
|   └───real/
|       ├───0046.jpg
|       └───...
└───test/
    ├───fake/
    │   ├───0046.jpg
    │   └───...
    └───real/
        ├───0046.jpg
        └───...
```



* Ejecuta las siguientes dos celdas para definir la ruta al conjunto de datos y visualizar algunas muestras de la carpeta de entrenamiento.

In [ ]:
# Cargar la ruta del conjunto de datos 
dataset_path = "./AIvsReal_sampled"  

# Analiza las divisiones del conjunto de datos en la ruta dada e imprime un recuento de imágenes para cada clase.
helper_utils.dataset_images_per_class(dataset_path)

**Nota**: Durante el resto de esta tarea, utilizarás el directorio `test` como tu **conjunto de validación**.

In [ ]:
# Seleccionar y mostrar aleatoriamente una cuadrícula de imágenes de muestra de la carpeta 'train'.
helper_utils.display_train_images(dataset_path)

<a name='1-1'></a>
### 1.1 - Explorando los archivos: Carga de datos con ImageFolder

Ahora que has revisado el conjunto de datos, tu próximo objetivo es cargar las imágenes en una estructura que PyTorch pueda utilizar para el entrenamiento. Para conjuntos de datos que están organizados nítidamente en subdirectorios para cada clase, como tus datos de `AIvsReal_sampled`, `ImageFolder` es una herramienta excepcionalmente conveniente y estándar.

Esta herramienta funciona infiriendo automáticamente las etiquetas de las clases a partir de los nombres de los subdirectorios. Espera que tus datos estén organizados de tal manera que `ruta_del_dataset/nombre_clase/nombre_imagen.jpg` sea una imagen perteneciente a `nombre_clase`. Esta característica se alinea perfectamente con la estructura organizada de tu conjunto de datos, ahorrándote un tiempo de preparación significativo.



Una ventaja fundamental de `ImageFolder` es su atributo `transform`. Este permite aplicar una secuencia de transformaciones de imagen, dándote un control preciso sobre cómo se preparan tus datos. Tienes la flexibilidad de establecer estas transformaciones cuando cargas los datos por primera vez o aplicarlas más tarde. Esta adaptabilidad asegura que tu flujo de carga de datos sea altamente eficaz y reutilizable para futuros proyectos.

<a name='ex-1'></a>
### Ejercicio 1 - create_dataset_splits

Tu primer ejercicio consiste en implementar la función `create_dataset_splits`. Esta función cargará tus datos de imagen desde el disco y creará dos objetos `Dataset` de PyTorch distintos.

**Tu tarea**:

* Usa la clase [ImageFolder](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html) para crear un objeto de conjunto de datos para tus datos de entrenamiento y otro para tus datos de validación.
* Las rutas para los directorios de entrenamiento y validación (`train_path` y `val_path`) ya han sido definidas para ti. Deberás pasar la ruta correcta a cada instancia de `ImageFolder`.
* La función debe devolver los dos conjuntos de datos que crees: `train_dataset` y `val_dataset`.

<details>
<summary><b><font color="green">Consejos adicionales de código (Haz clic para expandir si te has atascado)</font></b></summary>

Si necesitas ayuda, aquí tienes una guía más detallada.

Necesitas crear dos instancias de la clase `ImageFolder`. El argumento principal que debes proporcionar es `root`, que indica a `ImageFolder` dónde encontrar las imágenes.

* La primera línea para el conjunto de datos de entrenamiento se verá así:
    > `train_dataset = ImageFolder(root=train_path)`
* Ahora, aplica la misma lógica para crear el `val_dataset`. Asegúrate de usar la variable `val_path` para su `root`.

</details>

In [ ]:
# FUNCIÓN CALIFICADA: create_dataset_splits

def create_dataset_splits(data_path):
    """
    Crea conjuntos de datos de entrenamiento y validación a partir de una estructura 
    de directorios usando ImageFolder.

    Args:
        data_path (str): La ruta raíz al directorio del conjunto de datos, que debe
                         contener los subdirectorios 'train' y 'test'.

    Returns:
        tuple: Una tupla que contiene el train_dataset y validation_dataset
               (train_dataset, validation_dataset).
    """

    # Construye la ruta completa al directorio de datos de entrenamiento.
    train_path = data_path + "/train"
    # Construye la ruta completa al directorio de datos de validación.
    val_path = data_path + "/test"

    ### EMPIEZA TU CÓDIGO AQUÍ ###
    
    # Crea el conjunto de datos de entrenamiento usando ImageFolder
    train_dataset = None(
        # Establece la raíz (root) a la ruta del conjunto de datos de entrenamiento
        root=None,
    ) 

    # Crea el conjunto de datos de validación usando ImageFolder
    val_dataset = None(
        # Establece la raíz (root) a la ruta del conjunto de datos de validación
        root=None,
    ) 

    ### TERMINA TU CÓDIGO AQUÍ ###

    return train_dataset, val_dataset

In [ ]:
# Verify that the function loads the datasets
temp_train, temp_val = create_dataset_splits(dataset_path)

print("--- Training Dataset ---")
print(temp_train)
print("\n--- Validation Dataset ---")
print(temp_val)

#### Expected Output:
```
--- Training Dataset ---
Dataset ImageFolder
    Number of datapoints: 5000
    Root location: ../AIvsReal_sampled/train

--- Validation Dataset ---
Dataset ImageFolder
    Number of datapoints: 1000
    Root location: ../AIvsReal_sampled/test
```    

In [ ]:
# Test your code! 
unittests.exercise_1(create_dataset_splits)    

<a name='1-2'></a>
### 1.2 - Aumentos especializados: Protocolos para entrenamiento y validación

Ahora es el momento de definir las transformaciones que prepararán tus imágenes para la red neuronal. Estos pasos de preprocesamiento son fundamentales para estandarizar tus datos y para aumentar el conjunto de entrenamiento, lo que hará que tu modelo sea más robusto.

Cuando se utilizan arquitecturas preentrenadas, como harás más adelante, es una práctica estándar aplicar conjuntos de transformaciones *diferentes* a tus conjuntos de datos de entrenamiento y validación. Al aprovechar un modelo preentrenado, es **esencial** que las transformaciones básicas aplicadas tanto a tus datos de entrenamiento como a los de validación sean idénticas a las utilizadas para entrenar el modelo original. Esto es particularmente importante para el redimensionamiento de las imágenes y el uso de los mismos valores de normalización (media y desviación estándar). Aunque las técnicas de aumento de datos pueden variar entre tus datos de entrenamiento y validación, estos pasos fundamentales garantizan que tus datos se procesen de una manera que el modelo preentrenado ya comprenda.

* Antes de pasar a las transformaciones, primero definirás los valores de normalización estándar del conjunto de datos ImageNet, en el que se entrenaron muchos modelos preentrenados.

In [ ]:
# Define the standard mean values for the ImageNet dataset
imagenet_mean = torch.tensor([0.485, 0.456, 0.406])

# Define the standard standard deviation values for the ImageNet dataset
imagenet_std = torch.tensor([0.229, 0.224, 0.225])

<a name='ex-2'></a>
### Ejercicio 2 - define_transformations

En este ejercicio, implementarás la función `define_transformations` para crear dos flujos (pipelines) de transformación de imágenes distintos: uno para aumentar los datos de entrenamiento y otro para preparar los datos de validación.



**Tu tarea**:

* **Para el flujo `train_transform`:**
    * Debes encadenar una serie de transformaciones usando [transforms.Compose](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.Compose.html).
    * Este flujo debe incluir operaciones para:
        * [Redimensionamiento aleatorio](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.RandomResizedCrop.html).
        * [Volteo horizontal aleatorio](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.RandomHorizontalFlip.html).
        * [Ajustes de color](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.ColorJitter.html) para aumentar los datos.
        * [Conversión a tensor](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.ToTensor.html).
        * [Normalización](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html).
>
* **Para el flujo `val_transform`:**
    * Crea un segundo flujo con `transforms.Compose`.
    * Este será más sencillo y solo debe incluir los pasos esenciales:
        * [Redimensionar](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.Resize.html) la imagen.
        * [Conversión a tensor](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.ToTensor.html).
        * [Normalización](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html).
> 
* Ambos flujos deben utilizar los valores de `mean` (media) y `std` (desviación estándar) pasados a la función para la normalización.

<details>
<summary><b><font color="green">Consejos adicionales de código (Haz clic para expandir si te has atascado)</font></b></summary>

Si buscas una guía más detallada, sigue estos pasos.

**Para `train_transform`:**

Tu lista en `transforms.Compose` debe contener cinco pasos de transformación en este orden:
1. Agrega la transformación `RandomResizedCrop`, estableciendo el tamaño de salida en `(224, 224)`.
2. Agrega la transformación `RandomHorizontalFlip`.
3. Agrega la transformación `ColorJitter`, estableciendo `brightness` y `contrast` en `0.2`.
4. Agrega la transformación `ToTensor` para convertir imágenes a tensores.
5. Agrega la transformación `Normalize`, utilizando las variables `mean` y `std` de los argumentos de la función.

**Para `val_transform`:**

Tu lista `transforms.Compose` para validación es más simple y tiene tres pasos:
1. La primera transformación es para redimensionar:
    > `transforms.Resize((224, 224))`
2. Luego, agrega la transformación `ToTensor`.
3. Finalmente, agrega la transformación `Normalize`, usando nuevamente las variables `mean` y `std`.

</details>

In [ ]:
# FUNCIÓN CALIFICADA: define_transformations

def define_transformations(mean=imagenet_mean, std=imagenet_std):
    """
    Define series separadas de transformaciones de imagen para los conjuntos de datos de entrenamiento y validación.

    Args:
        mean (list o tuple): Los valores de la media (para cada canal, ej. RGB) calculados de ImageNet.
        std (list o tuple): Los valores de la desviación estándar (para cada canal) calculados de ImageNet.

    Returns:
        tuple: Una tupla que contiene dos objetos `torchvision.transforms.Compose`:
               - El primero para las transformaciones de entrenamiento.
               - El segundo para las transformaciones de validación.
    """

    ### EMPIEZA TU CÓDIGO AQUÍ ###

    # Crea un objeto Compose para encadenar múltiples transformaciones para el conjunto de entrenamiento
    
    # Inicializa 'train_transform' usando transforms.Compose para aplicar una secuencia de transformaciones
    train_transform = None.None([
        # Redimensiona y recorta aleatoriamente la imagen de entrada a 224x224 píxeles
        None,

        # Aplica un volteo horizontal aleatorio a la imagen para el aumento de datos
        None,

        # Cambia aleatoriamente el brillo y el contraste de la imagen para el aumento de datos
        # Establece `brightness=0.2` y `contrast=0.2`
        None,

        # Convierte la imagen PIL a un Tensor de PyTorch
        None,

        # Normaliza la imagen de tipo tensor con la 'mean' y 'std' proporcionadas
        None,
    ]) 

    # Crea un objeto Compose para encadenar múltiples transformaciones para el conjunto de validación
    
    # Inicializa 'val_transform' usando transforms.Compose para aplicar una secuencia de transformaciones
    val_transform = None.None([
        # Redimensiona la imagen de entrada a 224x224 píxeles
        None,

        # Convierte la imagen PIL a un Tensor de PyTorch
        None,

        # Normaliza la imagen de tipo tensor con la 'mean' y 'std' proporcionadas
        None,
    ]) 

    ### TERMINA TU CÓDIGO AQUÍ ###

    return train_transform, val_transform

In [ ]:
# Create the composed transformations
combined_transformations = define_transformations()

# Print the composed transformations to verify the sequence of operations
print("Augmented Training Transformations:\n")
print(combined_transformations[0])
print("\nValidation Transformations:\n")
print(combined_transformations[1])

#### Expected Output:
```
Augmented Training Transformations:

Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=None, hue=None)
    ToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)

Validation Transformations:

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)
```

In [ ]:
# Test your code! 
unittests.exercise_2(define_transformations)    

<a name='1-3'></a>
### 1.3 - Preparando la plataforma de lanzamiento de datos: Aplicando transformaciones y agrupando datos en lotes

Ya has gestionado la creación de tus objetos de conjunto de datos de entrenamiento y validación, y has definido por separado los flujos de transformación para cada uno. Una característica potente de los objetos `ImageFolder` es su atributo incorporado `.transform`. Ahora aplicarás las transformaciones que definiste asignando cada flujo a su respectivo conjunto de datos. Esto proporciona una forma limpia y modular de preparar cada subconjunto de datos para su propósito específico. Después de esto, usarás `DataLoader` para crear cargadores de datos barajados y agrupados por lotes (*batches*) para tus datos de entrenamiento y validación, lo cual es un paso esencial para prepararlos para el entrenamiento.

<a name='ex-3'></a>
### Ejercicio 3 - create_data_loaders

Tu tarea es implementar la función `create_data_loaders`. Esta función lo une todo aplicando las transformaciones a tus conjuntos de datos y luego envolviéndolos en objetos `DataLoader` para prepararlos para el entrenamiento.

**Tu tarea**:

* **Obtener las transformaciones**:
    * Llama a la función `define_transformations()` que implementaste anteriormente para obtener los flujos `train_transform` y `val_transform`.
>
* **Aplicar las transformaciones**:
    * Asigna el `train_transform` al atributo `.transform` del `trainset` de entrada.
    * Asigna el `val_transform` al atributo `.transform` del `valset` de entrada.
>
* **Crear los DataLoaders**:
    * Instancia un [`DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) para el conjunto de entrenamiento. Recuerda barajar (*shuffle*) los datos de entrenamiento.
    * Instancia un segundo `DataLoader` para el conjunto de validación. Este no debe ser barajado.
>
* Ambas instancias de `DataLoader` deben usar el `batch_size` (tamaño de lote) proporcionado a la función.

<details>
<summary><b><font color="green">Consejos adicionales de código (Haz clic para expandir si te has atascado)</font></b></summary>

Si necesitas una guía más detallada, sigue estos pasos.

1.  **Obtén las transformaciones**: Es una sola llamada a la función con dos valores de retorno.
    > `train_transform, val_transform = define_transformations()`

2.  **Aplica las transformaciones**: Solo necesitas asignar los objetos de transformación al atributo correcto del conjunto de datos. El primero es:
    > `trainset.transform = train_transform`
    > Ahora haz lo mismo para el `valset` usando el `val_transform`.

3.  **Crea el `DataLoader` de entrenamiento**: El `DataLoader` necesita el conjunto de datos, el `batch_size` y el estado de barajado.
    > `train_loader = crea un nuevo DataLoader usando (el trainset, el batch_size de la función, y establece shuffle en True)`

4.  **Crea el `DataLoader` de validación**: Este es casi idéntico al cargador de entrenamiento.
    > `val_loader = crea un nuevo DataLoader usando (el valset, el batch_size de la función, y establece shuffle en False)`

</details>

In [ ]:
# FUNCIÓN CALIFICADA: create_data_loaders

def create_data_loaders(trainset, valset, batch_size):
    """
    Crea instancias de DataLoader para los conjuntos de datos de entrenamiento y validación 
    con sus respectivas transformaciones.

    Args:
        trainset (torch.utils.data.Dataset): El conjunto de datos de entrenamiento.
        valset (torch.utils.data.Dataset): El conjunto de datos de validación.
        batch_size (int): El número de muestras a cargar en cada lote (batch).

    Returns:
        tuple: Una tupla que contiene:
            - train_loader (torch.utils.data.DataLoader): DataLoader para el conjunto de entrenamiento.
            - val_loader (torch.utils.data.DataLoader): DataLoader para el conjunto de validación.
            - trainset (torch.utils.data.Dataset): El conjunto de entrenamiento original con las transformaciones aplicadas.
            - valset (torch.utils.data.Dataset): El conjunto de validación original con las transformaciones aplicadas.
    """
    
    ### EMPIEZA TU CÓDIGO AQUÍ ###
    
    # Define transformaciones separadas para los conjuntos de entrenamiento y validación
    # Usa define_transformations() para obtener train_transform y val_transform
    train_transform, val_transform = None
    
    # Aplica las transformaciones de entrenamiento directamente al trainset estableciendo el atributo .transform
    None = None
    # Aplica las transformaciones de validación directamente al valset estableciendo el atributo .transform
    None = None
    
    # Crea un DataLoader para el conjunto de datos de entrenamiento
    # Usa el trainset transformado
    # Establece batch_size con el valor de entrada batch_size
    # Establece shuffle=True
    train_loader = None
    
    # Crea un DataLoader para el conjunto de datos de validación
    # Usa el valset transformado
    # Establece batch_size con el valor de entrada batch_size
    # Establece shuffle=False
    val_loader  = None
    
    ### TERMINA TU CÓDIGO AQUÍ ###
    
    return train_loader, val_loader, trainset, valset

In [ ]:
dataloaders = create_data_loaders(temp_train, temp_val, batch_size=16)

print("--- Train Loader ---")
helper_utils.display_data_loader_contents(dataloaders[0])
print("\n--- Val Loader ---")
helper_utils.display_data_loader_contents(dataloaders[1])

#### Expected Output:
```
--- Train Loader ---
Length: 313
--- Batch 1 ---
Data shape: torch.Size([16, 3, 224, 224])
Labels shape: torch.Size([16])

--- Val Loader ---
Length: 63
--- Batch 1 ---
Data shape: torch.Size([16, 3, 224, 224])
Labels shape: torch.Size([16])
```

In [ ]:
# Test your code! 
unittests.exercise_3(create_data_loaders)

<a name='2'></a>
## 2 - Expedición avanzada: Aprovechando la sabiduría de los modelos preentrenados

Has preparado tus datos. Ahora es el momento del corazón de esta expedición: seleccionar y trabajar con un **modelo preentrenado**. Este enfoque, conocido como **aprendizaje por transferencia (transfer learning)**, es una piedra angular de la visión artificial moderna.



En lugar de construir un modelo desde cero, puedes aprovechar uno que ya ha aprendido de un conjunto de datos masivo y diverso, como ImageNet. Piénsalo como el uso de una herramienta potente y veterana, ya equipada con una comprensión profunda de los patrones visuales generales.

Los beneficios de este enfoque son sustanciales:

* **Descubrimiento acelerado**: Obtienes una ventaja significativa, ya que el modelo ya ha aprendido un conjunto rico de características.
* **Rendimiento superior**: Los modelos preentrenados a menudo logran una mayor precisión, especialmente cuando tu conjunto de datos es pequeño.

<a name='2-1'></a>
### 2.1 - Seleccionando a tu campeón: Presentando MobileNetV3-Large

Tu primera tarea en esta fase es elegir una arquitectura preentrenada adecuada. Para esta tarea, trabajarás con [MobileNetV3-Large](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.mobilenet_v3_large.html), un modelo eficiente y potente.



**¿Por qué MobileNetV3-Large?**

Es una excelente elección para una tarea como la identificación de imágenes generadas por IA debido a su:

* **Alta eficiencia**: Ofrece un equilibrio excepcional entre precisión y uso de recursos, lo que lo hace muy adecuado para el rendimiento en entornos con recursos limitados.
* **Sólida precisión de clasificación**: A pesar de su enfoque en la eficiencia, MobileNetV3-Large ofrece un rendimiento robusto, incorporando elementos de diseño avanzados optimizados mediante la Búsqueda de Arquitectura Neuronal (NAS).
* **Extracción de características eficaz**: Al haber sido preentrenado en el masivo conjunto de datos ImageNet, el modelo ya ha aprendido una jerarquía rica de características visuales, una base sólida que puedes adaptar para distinguir entre imágenes reales y falsas.

<a name='ex-41'></a>
### Ejercicio 4.1 - load_mobilenetv3_model

Tu tarea es implementar la función `load_mobilenetv3_model`. En esta tarea, en lugar de descargar los pesos de Internet, los cargarás desde un archivo local que se te ha proporcionado.

**Tu tarea**:

* **Cargar la arquitectura del modelo**:
    * Instancia un modelo [**`mobilenet_v3_large`**](https://pytorch.org/vision/main/models/generated/torchvision.models.mobilenet_v3_large.html) de `torchvision.models`.
    * Es esencial inicializarlo **sin** sus pesos preentrenados (`weights=None`) por ahora (los cargarás en el siguiente paso).
>
* **Cargar los pesos**:
    * Usa **`torch.load()`** para cargar el diccionario de estados del modelo (sus pesos) desde la ruta `weights_path` proporcionada a la función.
>
* El resto de la función, que aplica estos pesos cargados a la arquitectura del modelo, ya está completada para ti.

<details>
<summary><b><font color="green">Consejos adicionales de código (Haz clic para expandir si te has atascado)</font></b></summary>

Si necesitas una guía más detallada, aquí tienes los pasos.

1.  **Carga la arquitectura del modelo**: Debes llamar a la función `mobilenet_v3_large`. Para asegurar que no descargue ningún peso, debes establecer el argumento `weights` en `None`.
    > `model = tv_models.mobilenet_v3_large(weights=None)`

2.  **Carga el archivo de pesos local**: Usa la función `torch.load()` para cargar los pesos. Debes proporcionar la ruta del archivo y un `map_location` para asegurar que funcione en cualquier dispositivo (CPU o GPU).
    > `state_dict = carga el modelo torch desde (el weights_path, y establece map_location en un dispositivo cpu de torch)`

</details>

In [ ]:
# FUNCIÓN CALIFICADA: load_mobilenetv3_model

def load_mobilenetv3_model(weights_path):
    """
    Carga un modelo MobileNetV3-Large preentrenado de torchvision.

    Args:
        weights_path (str): La ruta del archivo a los pesos del modelo .pth guardados.
        
    Returns:
        torch.nn.Module: Un modelo MobileNetV3-Large preentrenado.
    """
    ### EMPIEZA TU CÓDIGO AQUÍ ###

    # Carga el modelo MobileNetV3-Large sin pesos preentrenados.
    model = None.None

    # Carga el diccionario de estados (pesos) desde el archivo local.
    state_dict = None(None, map_location=torch.device('cpu'))

    ### TERMINA TU CÓDIGO AQUÍ ###
    
    model.load_state_dict(state_dict)

    return model

In [ ]:
# Cargar el modelo MobileNetV3-Large preentrenado usando los pesos del archivo local.
local_weights = "./mobilenet_weights/mobilenet_v3_large-8738ca79.pth"
test_model = load_mobilenetv3_model(local_weights)

# Imprimir la última capa del clasificador del modelo cargado
print(test_model.classifier[-1])

#### Expected Output:
```
Linear(in_features=1280, out_features=1000, bias=True)
```

In [ ]:
# Test your code! 
unittests.exercise_4_1(load_mobilenetv3_model)    

<a name='2-2'></a>
### 2.2 - Adaptando a tu campeón: Ajustando MobileNetV3 para la detección de falsificaciones

Has seleccionado a tu campeón: el modelo preentrenado `MobileNetV3-Large`. Aunque es un veterano en la clasificación de imágenes del vasto conjunto de datos ImageNet (con 1,000 clases), necesita ser adaptado para tu misión específica de identificar imágenes generadas por IA, que solo tiene dos clases. Este proceso se llama **extracción de características** (feature extraction).

Para adaptar el modelo, seguirás dos pasos fundamentales:

* **Congelar las capas centrales**: Las capas iniciales del modelo, responsables de extraer patrones visuales generales como bordes, texturas y formas, contienen un conocimiento inestimable. Al **congelar** estas capas, evitas que sus pesos se actualicen durante el entrenamiento. Esto preserva la base potente del modelo, reduce significativamente el número de parámetros entrenables y acelera el entrenamiento.



* **Reemplazar el cabezal clasificador**: El clasificador final del modelo original está configurado para 1,000 clases y no es adecuado para tu tarea de dos clases. Lo reemplazarás con un nuevo clasificador personalizado. Esta nueva capa tomará las características de alto nivel del núcleo congelado y generará una predicción para tus dos clases objetivo, preparando a tu modelo para convertirse en un FakeFinder especializado.

<a name='ex-42'></a>
### Ejercicio 4.2 - update_model_last_layer

Tu última tarea calificada es implementar la función `update_model_last_layer`. Este es el núcleo de la técnica de **extracción de características**, donde adaptarás el modelo preentrenado para tu tarea específica de clasificación `AIvsReal`.

**Tu tarea**:

* **Congelar las capas de características**:
    * El conocimiento preentrenado del modelo reside en sus capas de características (`model.features`). Necesitas **congelar** estas capas para evitar que se actualicen durante el entrenamiento.
    * Para hacer esto, iterarás a través de todos los parámetros en `model.features.parameters()` y establecerás su atributo `requires_grad` en `False`.
>
* **Reemplazar el cabezal clasificador**:
    * El clasificador original (`model.classifier[-1]`) está diseñado para 1,000 clases. Debes reemplazarlo por uno nuevo adecuado para tu tarea.
    * Necesitarás obtener el número de características de entrada del clasificador original.
    * Luego, crea una nueva capa [nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) con el número correcto de características de entrada y el nuevo `num_classes` como salida.
    * Finalmente, reemplaza el antiguo clasificador por el nuevo.
>

<details>
<summary><b><font color="green">Consejos adicionales de código (Haz clic para expandir si te has atascado)</font></b></summary>

Si buscas una guía más detallada, sigue estos pasos.

**1. Congelar las capas**:
Esto requiere un bucle `for`.
> `for cada parámetro en los parámetros de características del modelo:`
> &nbsp;&nbsp;&nbsp;&nbsp;`establece el atributo requires_grad del parámetro en False`

**2. Reemplazar el clasificador**:
Este es un proceso de tres pasos para la segunda mitad de la función.

* Primero, obtén el número de características de entrada de la última capa del clasificador original del modelo.
    > `num_features = obtén el in_features de model.classifier[-1]`

* A continuación, crea la nueva capa lineal. Necesita el `num_features` que acabas de recuperar y el `num_classes` de los argumentos de la función.
    > `new_classifier = crea una nueva capa nn.Linear con (in_features establecido en num_features, y out_features en num_classes)`

* Finalmente, reemplaza la capa antigua con la nueva.
    > `model.classifier[-1] = tu nuevo_classifier`

</details>

In [ ]:
# FUNCIÓN CALIFICADA: update_model_last_layer

def update_model_last_layer(model, num_classes):
    """
    Congela las capas de características de un modelo preentrenado y reemplaza su capa de
    clasificación final por una nueva adaptada al número especificado de clases.

    Args:
        model (torch.nn.Module): El modelo preentrenado a modificar.
        num_classes (int): El número de clases de salida para la nueva capa de clasificación.

    Returns:
        torch.nn.Module: El modelo modificado con las capas de características congeladas y una nueva
                         capa de clasificación.
    """
    ### EMPIEZA TU CÓDIGO AQUÍ ###

    # Congela los parámetros de las capas de características del modelo
    # Itera a través de cada parámetro en model.features.parameters()
    for None in None():
        # Establece el atributo requires_grad de cada feature_parameter en False
        feature_parameter.requires_grad = None

    # Accede a la capa de clasificación final del modelo
    last_classifier_layer = model.classifier[-1] 
    
    # Accede al atributo in_features de last_classifier_layer
    num_features = None
    
    # Usa nn.Linear para crear una nueva capa lineal para la clasificación con el número original de
    # características de entrada y el número especificado de clases de salida
    new_classifier = None(in_features=None, out_features=None)
    
    # Reemplaza la capa de clasificación final original con la capa recién creada
    model.classifier[-1] = None
    
    ### TERMINA TU CÓDIGO AQUÍ ###

    return model

In [ ]:
# Modificar la última capa del modelo MobileNetV3-Large
test_model = update_model_last_layer(test_model, num_classes=5)

# Imprimir la última capa del clasificador del modelo modificado
print(test_model.classifier[-1])

#### Expected Output:
```
Linear(in_features=1280, out_features=5, bias=True)
```

In [ ]:
# Test your code! 
unittests.exercise_4_2(update_model_last_layer)    

---
# Submission Note

Congratulations! You've completed the final graded exercise of this assignment.

If you've successfully passed all the unit tests above, you've completed the core requirements of this assignment. Feel free to [submit](#submission) your work now. The grading process runs in the background, so it will not disrupt your progress and you can continue on with the rest of the material.

**🚨 IMPORTANT NOTE** If you have passed all tests within the notebook, but the autograder shows a system error after you submit your work:

<div style="background-color: #1C1C1E; border: 1px solid #444444; color: #FFFFFF; padding: 15px; border-radius: 5px;">
    <p><strong>Grader Error: Grader feedback not found</strong></p>
    <p>Autograder failed to produce the feedback...</p>
</div>
<br>

This is typically a temporary system glitch. The most common solution is to resubmit your assignment, as this often resolves the problem. Occasionally, it may be necessary to resubmit more than once. 
>
If the error persists, please reach out for support in the [DeepLearning.AI Community Forum](https://community.deeplearning.ai/c/course-q-a/pytorch-for-developers/pytorch-techniques-and-ecosystem-tools/561).

---

<a name='3'></a>
## 3 - Expedición final: Entrenando y evaluando a tu campeón adaptado

Esta sección te permite llevar a tu modelo MobileNetV3 adaptado a través de un ciclo de entrenamiento y observar su rendimiento. Es una oportunidad para ver cómo el conocimiento aprovechado de un modelo preentrenado, combinado con tus técnicas especializadas, da lugar a un potente FakeFinder.



<a name='3-1'></a>
### 3.1 - Configurando la plataforma de lanzamiento: Preparando los DataLoaders y el modelo adaptado

Antes de poder comenzar el entrenamiento real, primero debes configurar tu entorno ejecutando tres pasos fundamentales:

* **Inicialización de los conjuntos de datos**: Primero cargarás tus datos de entrenamiento y validación desde sus respectivos directorios.

* **Inicialización de los DataLoaders**: A continuación, crearás el `train_loader` y el `val_loader` para alimentar al modelo con tus imágenes transformadas en lotes manejables.

* **Preparación del modelo**: Cargarás el modelo MobileNetV3-Large preentrenado y luego adaptarás su capa de clasificación final para tus dos clases específicas. Esto prepara a tu modelo campeón para su misión especializada.

La siguiente celda de código ejecutará estos pasos preparatorios, asegurando que tanto tus datos como tu modelo estén listos para el bucle de entrenamiento.

In [ ]:
# Inicializar los conjuntos de datos de entrenamiento y validación
train_dataset, val_dataset = create_dataset_splits(dataset_path)

# Inicializar los dataloaders para entrenamiento y validación
train_loader, val_loader, _, __ = create_data_loaders(train_dataset, val_dataset, batch_size=32)

In [ ]:
# Cargar el modelo MobileNetV3-Large preentrenado y modificar su última capa
local_weights = "./mobilenet_weights/mobilenet_v3_large-8738ca79.pth"
mobilenet_model = load_mobilenetv3_model(local_weights)
mobilenet_model = update_model_last_layer(mobilenet_model, num_classes=2)

<a name='3-2'></a>
### 3.2 - Entrenamiento del modelo adaptado

Con tus datos preparados y el modelo MobileNetV3 adaptado, es hora de ajustar sus capas con tu conjunto de datos de imágenes de IA frente a imágenes reales. Este paso de entrenamiento preparará al modelo para convertirse en un FakeFinder especializado.

* Define la función de pérdida y el optimizador.
    * El optimizador solo actualizará las capas no congeladas del modelo, que en este caso, es solo el nuevo cabezal clasificador.

In [ ]:
# Define la función de pérdida para calcular la diferencia entre la salida del modelo y las etiquetas verdaderas
loss_fcn = nn.CrossEntropyLoss()

# Define el optimizador para actualizar los pesos del modelo durante el entrenamiento
optimizer = optim.Adam(filter(lambda p: p.requires_grad, 
                             mobilenet_model.parameters()), 
                       lr=0.001)

* Define el número de épocas para el entrenamiento.
    * Puedes experimentar ajustando este número para observar diferentes resultados de entrenamiento, pero por ahora, ejecútalo durante solo una época para ver los potentes resultados de un modelo preentrenado.

In [ ]:
# EDITABLE CELL:

#Establece el número de épocas
num_epochs = 1

* Usa la función auxiliar `training_loop` para gestionar el ciclo estándar de entrenamiento y validación, un proceso que ya debería resultarte familiar.


In [ ]:
# Train the model
trained_model = helper_utils.training_loop(
    mobilenet_model, 
    train_loader, 
    val_loader,
    loss_fcn,
    optimizer,
    DEVICE, 
    num_epochs
)

<br>

Con el entrenamiento completado, es hora de reflexionar sobre las notables mejoras que acabas de observar.

En tu tarea anterior, navegaste por un complejo paisaje de hiperparámetros con Optuna para encontrar una configuración de modelo personalizada. Tu modelo con mejor rendimiento logró una sólida precisión de validación de más del **70%** después de **3 épocas** de entrenamiento. Ese fue un excelente resultado nacido de un proceso de búsqueda meticuloso y que requirió mucho tiempo.

Ahora, simplemente aprovechando un modelo MobileNetV3-Large preentrenado y realizando la extracción de características, ¡has alcanzado una precisión y exactitud de validación de más del **80%** en solo **1 época**! Este es un aumento de rendimiento dramático en una fracción del tiempo.

Esta diferencia ilustra de manera poderosa una lección fundamental en el aprendizaje automático: la extensa base de conocimientos de un modelo preentrenado proporciona un camino acelerado hacia un rendimiento superior. Si bien la construcción de arquitecturas personalizadas es fundamental para comprender cómo funciona el aprendizaje profundo, saber cuándo y cómo aplicar la sabiduría de los modelos preentrenados es la forma de conquistar misiones complejas del mundo real con una velocidad y eficiencia sin precedentes.

<a name='3-3'></a>
### 3.3 - Revisión de rendimiento: Visualización de predicciones

Tu modelo adaptado ha completado su entrenamiento. Aunque la puntuación de precisión final te proporciona una medida cuantitativa de su rendimiento, a menudo es más revelador ver al modelo en acción. En esta sección final, realizarás una revisión cualitativa de tu FakeFinder mejorado.

Primero, visualizarás las predicciones del modelo en un lote aleatorio de imágenes del conjunto de validación. Esto te ayudará a tener una idea de dónde tiene éxito el modelo y dónde podría cometer errores, proporcionando una imagen más clara que los números por sí solos.

* Ejecuta la celda de abajo para mostrar una cuadrícula de imágenes del conjunto de validación, mostrando la etiqueta real y la predicción del modelo para cada una.

In [ ]:
# Obtener la lista de nombres de clases ('fake', 'real') del conjunto de datos de validación.
class_names = val_dataset.classes

# Visualizar las predicciones realizadas por el modelo entrenado
helper_utils.visualize_predictions(trained_model, val_loader, DEVICE, class_names)

#### Pon a prueba tu FakeFinder con una imagen nueva

Ahora es el momento de poner a prueba tu FakeFinder con un nuevo dato. Puedes subir tu propia imagen para ver cómo la clasifica el modelo. Esta es una excelente manera de evaluar su rendimiento con datos que nunca ha visto.

Ejecuta la función `helper_utils.upload_jpg_widget()`, la cual muestra un widget para subir imágenes directamente a tu espacio de trabajo. Por favor, ten en cuenta lo siguiente:

* Solo puedes subir imágenes que tengan la extensión **.jpg**.
* El tamaño del archivo de la imagen no debe exceder los **5 MB**.
* Después de una carga exitosa, se mostrará la ruta del archivo de la imagen. Debes copiar esta ruta para el siguiente paso.
* Una vez que se muestre el widget, puedes usarlo varias veces para subir imágenes; no es necesario volver a ejecutar la función `helper_utils.upload_jpg_widget()`.

In [ ]:
helper_utils.upload_jpg_widget()

* Establece la ruta a tu imagen (como se mostró anteriormente).

Alternativamente, puedes usar estas imágenes que ya están presentes en el espacio de trabajo:
> * `image_path = './images/fake/birds_sheep_dog.jpg'`
> * `image_path = './images/fake/car_bus_tram.jpg'`
> * `image_path = './images/fake/person_and_bicycle.jpg'`
> * `image_path = './images/real/eiffel_tower.jpg'`
> * `image_path = './images/real/minar.jpg'`
> * `image_path = './images/real/statue_liberty.jpg'`


* Se ha establecido una ruta por defecto para ti, pero siéntete libre de cambiarla por una diferente.

In [ ]:
# EDITABLE CELL:

image_path = './images/fake/birds_sheep_dog.jpg' ### Agrega la ruta de tu imagen aquí

In [ ]:
# Display a prediction for the single uploaded image.
helper_utils.make_predictions(trained_model, image_path, DEVICE, class_names)

## Conclusión

¡Felicidades, Explorador de IA! Has completado con éxito tu segunda expedición y has mejorado significativamente tu FakeFinder. Al aplicar magistralmente el **aprendizaje por transferencia** (transfer learning), no solo has mejorado el rendimiento de tu modelo, sino que también has añadido a tu conjunto de habilidades una de las técnicas más prácticas y potentes de la IA moderna.


En esta tarea, cargaste con éxito un modelo MobileNetV3 preentrenado, preservaste su conocimiento congelando sus capas centrales y reemplazaste hábilmente su clasificador para especializarlo en tu labor. Luego, entrenaste el modelo adaptado y fuiste testigo de un aumento dramático en el rendimiento, logrando una mayor precisión de validación en una fracción del tiempo de entrenamiento en comparación con tu modelo construido a medida.

Tu viaje a través de estas dos tareas ilustra una lección fundamental en la IA aplicada. En la primera expedición, aprendiste los principios esenciales para construir y optimizar un modelo desde cero. En esta segunda expedición, aprendiste a apoyarte en hombros de gigantes aprovechando modelos preentrenados para resolver problemas con una velocidad y eficiencia increíbles. Un practicante de IA verdaderamente hábil sabe cuándo construir desde la base y cuándo adaptar las herramientas existentes. Ahora estás equipado con ambos conjuntos de habilidades.

Bien hecho por completar una misión exitosa. Que tus futuras expediciones en el mundo de la IA sean aún más ambiciosas y gratificantes.